In [37]:
from __future__ import annotations
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Tuple, Optional
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader,TensorDataset
import zipfile
import requests
import io
print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

PyTorch version: 2.10.0+cu128
Using device: cuda


## Rishi's part
data cleaning and preprocessing

In [10]:
rm -rf 'deep_learning_project'

In [11]:
!git clone https://github.com/samarthsingh1/deep_learning_project


Cloning into 'deep_learning_project'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 116 (delta 29), reused 11 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 892.04 KiB | 3.39 MiB/s, done.
Resolving deltas: 100% (29/29), done.


In [12]:
!jupyter nbconvert --to script deep_learning_project/experimentation/data_utils.ipynb

[NbConvertApp] Converting notebook deep_learning_project/experimentation/data_utils.ipynb to script
[NbConvertApp] Writing 5212 bytes to deep_learning_project/experimentation/data_utils.py


In [13]:
%run deep_learning_project/experimentation/data_utils.py

(140256, 370)
False
41437378
              mean         max
MT_001    3.970785   48.223350
MT_002   20.768480  115.220484
MT_003    2.918308  151.172893
MT_004   82.184490  321.138211
MT_005   37.240309  150.000000
MT_006  141.227385  535.714286
MT_007    4.521338   44.657999
MT_008  191.401476  552.188552
MT_009   39.975354  157.342657
MT_010   42.205152  198.924731
                        total_load  hour_of_day  day_of_week  is_weekend  \
datetime                                                                   
2011-01-01 00:00:00  207058.270272            0            5           1   
2011-01-01 01:00:00  265378.510747            1            5           1   
2011-01-01 02:00:00  263924.219533            2            5           1   
2011-01-01 03:00:00  266306.134264            3            5           1   
2011-01-01 04:00:00  259854.210701            4            5           1   

                     month  hour_sin  hour_cos  dayofweek_sin  dayofweek_cos  \
datetime         

/content/deep_learning_project/experimentation/data_utils.py:65: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df.isna().sum()[0]==1
/content/deep_learning_project/experimentation/data_utils.py:73: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df["total_load"].resample("H").sum().to_frame()


In [39]:
# =========================================================
# 1. SEQUENCE CREATION
# =========================================================

INPUT_WINDOW = 168
FORECAST_HORIZON = 24
BATCH_SIZE = 64

FEATURE_COLS = [
    'total_load',
    'hour_sin', 'hour_cos',
    'dayofweek_sin', 'dayofweek_cos',
    'month_sin', 'month_cos'
]


def create_sequences(data, input_window=INPUT_WINDOW, forecast_horizon=FORECAST_HORIZON):
    features = data[FEATURE_COLS].values
    targets = data['total_load'].values

    X, y = [], []
    for i in range(len(features) - input_window - forecast_horizon + 1):
        X.append(features[i:i + input_window])
        y.append(targets[i + input_window:i + input_window + forecast_horizon])

    return (
        torch.tensor(np.array(X), dtype=torch.float32),
        torch.tensor(np.array(y), dtype=torch.float32)
    )


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# 2. CREATE TRAIN / VAL / TEST SEQUENCES
# =========================================================

X_train, y_train = create_sequences(train_scaled)
X_val, y_val = create_sequences(val_scaled)
X_test, y_test = create_sequences(test_scaled)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape},   y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")

# Store actual test targets for future use
y_test_scaled = y_test.numpy()


# =========================================================
# 3. NAIVE + SEASONAL FORECASTS ON TEST SET
# =========================================================

def generate_persistence_forecast(X_test, forecast_horizon=FORECAST_HORIZON):
    """Naive persistence: next 24h = last 24h of input window."""
    naive_persistence_preds_scaled = X_test[:, -forecast_horizon:, 0].numpy()
    return naive_persistence_preds_scaled


def seasonal_naive_forecast(X_context_scaled, forecast_horizon, seasonal_lag):
    """
    Seasonal naive forecast.
    For forecast step h, copy value from (-seasonal_lag + h) of input window.
    """
    load_col = X_context_scaled[:, :, 0].numpy()
    preds = np.zeros((load_col.shape[0], forecast_horizon))

    for h in range(forecast_horizon):
        idx = -seasonal_lag + h
        preds[:, h] = load_col[:, idx]

    return preds


# Naive persistence forecast on test set
naive_persistence_preds_scaled = generate_persistence_forecast(
    X_test, forecast_horizon=FORECAST_HORIZON
)

# Seasonal naive day forecast on test set
seasonal_naive_day_preds_scaled = seasonal_naive_forecast(
    X_test, FORECAST_HORIZON, seasonal_lag=24
)

# Seasonal naive week forecast on test set
seasonal_naive_week_preds_scaled = seasonal_naive_forecast(
    X_test, FORECAST_HORIZON, seasonal_lag=168
)


# =========================================================
# 4. LSTM MODELS
# =========================================================

class LSTMForecaster(nn.Module):
    """Univariate LSTM."""
    def __init__(self, input_size=1, hidden_size=64, num_layers=2,
                 dropout=0.1, forecast_horizon=24):
        super().__init__()
        lstm_dropout = dropout if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=lstm_dropout,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, forecast_horizon)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]
        last_hidden = self.dropout(last_hidden)
        return self.fc(last_hidden)


class MultiFeatureLSTMForecaster(nn.Module):
    """Multivariate LSTM."""
    def __init__(self, input_size=7, hidden_size=64, num_layers=2,
                 dropout=0.1, forecast_horizon=24):
        super().__init__()
        lstm_dropout = dropout if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=lstm_dropout,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, forecast_horizon)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]
        last_hidden = self.dropout(last_hidden)
        return self.fc(last_hidden)


def train_one_epoch(model, loader, optimizer, criterion, dev):
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(dev), y_batch.to(dev)

        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate_loss(model, loader, criterion, dev):
    model.eval()
    total_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(dev), y_batch.to(dev)
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def predict_model(model, loader, dev):
    """Return predictions and targets from a dataloader."""
    model.eval()
    preds_all, y_all = [], []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(dev)
        preds = model(X_batch).cpu().numpy()
        preds_all.append(preds)
        y_all.append(y_batch.numpy())

    return np.vstack(preds_all), np.vstack(y_all)


def train_lstm(model, train_loader, val_loader, num_epochs=20, lr=1e-3,
               patience=5, dev='cpu', label="LSTM"):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_state = None
    wait = 0

    print(f"\n===== Training {label} =====")
    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, dev)
        val_loss = evaluate_loss(model, val_loader, criterion, dev)

        print(f"{label} | Epoch {epoch:02d}/{num_epochs} | "
              f"Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            print(f"{label} | Early stopping at epoch {epoch}.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"{label} | Best Val Loss: {best_val_loss:.6f}")
    return model


# =========================================================
# 5. UNIVARIATE LSTM PREDICTIONS ON TEST SET
# =========================================================

set_seed(42)

X_train_uni = X_train[:, :, 0:1]
X_val_uni = X_val[:, :, 0:1]
X_test_uni = X_test[:, :, 0:1]

train_loader_uni = DataLoader(TensorDataset(X_train_uni, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader_uni = DataLoader(TensorDataset(X_val_uni, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader_uni = DataLoader(TensorDataset(X_test_uni, y_test), batch_size=BATCH_SIZE, shuffle=False)

model_uni = LSTMForecaster(
    input_size=1,
    hidden_size=64,
    num_layers=2,
    dropout=0.1,
    forecast_horizon=FORECAST_HORIZON
).to(device)

model_uni = train_lstm(
    model_uni,
    train_loader_uni,
    val_loader_uni,
    num_epochs=20,
    lr=1e-3,
    patience=5,
    dev=device,
    label="Univariate LSTM"
)

univariate_lstm_preds_scaled, univariate_lstm_targets_scaled = predict_model(
    model_uni, test_loader_uni, device
)


# =========================================================
# 6. MULTIVARIATE LSTM PREDICTIONS ON TEST SET
# =========================================================

set_seed(42)

train_loader_multi = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader_multi = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader_multi = DataLoader(TensorDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

model_multi = MultiFeatureLSTMForecaster(
    input_size=len(FEATURE_COLS),
    hidden_size=64,
    num_layers=2,
    dropout=0.1,
    forecast_horizon=FORECAST_HORIZON
).to(device)

model_multi = train_lstm(
    model_multi,
    train_loader_multi,
    val_loader_multi,
    num_epochs=20,
    lr=1e-3,
    patience=5,
    dev=device,
    label="Multivariate LSTM"
)

multivariate_lstm_preds_scaled, multivariate_lstm_targets_scaled = predict_model(
    model_multi, test_loader_multi, device
)


# =========================================================
# 7. STORED OUTPUT VARIABLES FOR FUTURE USE
# =========================================================

print("\nStored prediction variables ready for future use:")

print("Actual test targets:")
print(" - y_test_scaled")

print("\nNaive / Seasonal predictions:")
print(" - naive_persistence_preds_scaled")
print(" - seasonal_naive_day_preds_scaled")
print(" - seasonal_naive_week_preds_scaled")

print("\nLSTM predictions:")
print(" - univariate_lstm_preds_scaled")
print(" - multivariate_lstm_preds_scaled")

print("\nOptional matching target arrays returned alongside LSTM predictions:")
print(" - univariate_lstm_targets_scaled")
print(" - multivariate_lstm_targets_scaled")

X_train: torch.Size([24354, 168, 7]), y_train: torch.Size([24354, 24])
X_val:   torch.Size([5068, 168, 7]),   y_val:   torch.Size([5068, 24])
X_test:  torch.Size([5070, 168, 7]),  y_test:  torch.Size([5070, 24])

===== Training Univariate LSTM =====
Univariate LSTM | Epoch 01/20 | Train Loss: 0.183404 | Val Loss: 0.033081
Univariate LSTM | Epoch 02/20 | Train Loss: 0.037266 | Val Loss: 0.026921
Univariate LSTM | Epoch 03/20 | Train Loss: 0.032014 | Val Loss: 0.025289
Univariate LSTM | Epoch 04/20 | Train Loss: 0.029613 | Val Loss: 0.024240
Univariate LSTM | Epoch 05/20 | Train Loss: 0.028333 | Val Loss: 0.024079
Univariate LSTM | Epoch 06/20 | Train Loss: 0.027604 | Val Loss: 0.026595
Univariate LSTM | Epoch 07/20 | Train Loss: 0.026728 | Val Loss: 0.022056
Univariate LSTM | Epoch 08/20 | Train Loss: 0.026117 | Val Loss: 0.021634
Univariate LSTM | Epoch 09/20 | Train Loss: 0.025976 | Val Loss: 0.021590
Univariate LSTM | Epoch 10/20 | Train Loss: 0.025367 | Val Loss: 0.021193
Univariate